In [1]:
# ================================================================
# MODEL 3 — CHANGE-AWARE IMAGE CAPTIONING
#
# Task:
#   Given two bi-temporal remote sensing images,
#   generate a natural-language description of the change.
#
# Dataset:
#   LEVIR-CC
#
# Architecture:
#   Siamese CNN encoder
#   Temporal difference representation
#   GRU caption decoder
#
# Input:
#   Image A + Image B
#
# Output:
#   Natural-language change caption
#
# Evaluation:
#   BLEU-1
#   BLEU-2
#   BLEU-3
#   BLEU-4
#   METEOR
#   ROUGE-L
#
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

import os
import json
import ast
import re
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from PIL import Image

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from torchvision import models
from torchvision.models import ResNet18_Weights

from sklearn.metrics import accuracy_score

import nltk

from nltk.translate.bleu_score import (
    sentence_bleu,
    corpus_bleu,
    SmoothingFunction
)

from nltk.translate.meteor_score import (
    meteor_score
)

from rouge_score import rouge_scorer


# ================================================================
# 2. NLTK RESOURCE CHECK
# ================================================================

try:
    nltk.data.find(
        "corpora/wordnet"
    )
except LookupError:

    nltk.download(
        "wordnet"
    )

try:
    nltk.data.find(
        "corpora/omw-1.4"
    )
except LookupError:

    nltk.download(
        "omw-1.4"
    )


# ================================================================
# 3. CONFIGURATION
# ================================================================

CSV_PATH = Path(
    r"C:\College\Dissertation\Data\FINAL_LEVIR_DATASET\final_dataset.csv"
)

OUTPUT_DIR = Path(
    r"C:\College\Dissertation\Data\FINAL_LEVIR_DATASET"
) / "CAPTION_GENERATION_MODEL"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


BEST_MODEL_PATH = (
    OUTPUT_DIR /
    "model3_best_caption_model.pth"
)

HISTORY_PATH = (
    OUTPUT_DIR /
    "model3_training_history.csv"
)

VOCAB_PATH = (
    OUTPUT_DIR /
    "model3_vocabulary.json"
)

RESULTS_PATH = (
    OUTPUT_DIR /
    "model3_validation_predictions.csv"
)

REPORT_PATH = (
    OUTPUT_DIR /
    "model3_evaluation_report.txt"
)

CONFIG_PATH = (
    OUTPUT_DIR /
    "model3_config.json"
)


# ================================================================
# TRAINING PARAMETERS
# ================================================================

BATCH_SIZE = 16

EPOCHS = 20

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

MAX_CAPTION_LEN = 40

MIN_WORD_FREQ = 1

HIDDEN_DIM = 256

IMAGE_SIZE = 224

NUM_WORKERS = 0

PATIENCE = 6

SEED = 42


# ================================================================
# 4. REPRODUCIBILITY
# ================================================================

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )

torch.backends.cudnn.benchmark = True


# ================================================================
# 5. DEVICE
# ================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 72)
print("MODEL 3 — CHANGE-AWARE IMAGE CAPTIONING")
print("=" * 72)

print(
    f"Device: {DEVICE}"
)

if DEVICE.type == "cuda":

    print(
        f"GPU: {torch.cuda.get_device_name(0)}"
    )

    print(
        f"GPU Memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )


# ================================================================
# 6. LOAD DATA
# ================================================================

print()
print("=" * 72)
print("LOADING DATASET")
print("=" * 72)

df = pd.read_csv(
    CSV_PATH
)

print(
    f"Total rows: {len(df):,}"
)

print(
    df["split"].value_counts()
)


# ================================================================
# 7. REMOVE DUPLICATE CC PAIRS
# ================================================================

df["pair_id"] = (
    df["cc_A"].astype(str)
    + "||"
    + df["cc_B"].astype(str)
)

before = len(df)

df = df.drop_duplicates(
    subset="pair_id"
).copy()

print(
    f"Unique CC pairs: {len(df):,}"
)

print(
    f"Duplicates removed: "
    f"{before - len(df):,}"
)


# ================================================================
# 8. TRAIN / VALIDATION
# ================================================================

train_df = df[
    df["split"].astype(str).str.lower()
    == "train"
].reset_index(drop=True)

val_df = df[
    df["split"].astype(str).str.lower()
    == "val"
].reset_index(drop=True)

print(
    f"Train: {len(train_df):,}"
)

print(
    f"Validation: {len(val_df):,}"
)


# ================================================================
# 9. CAPTION PARSER
# ================================================================

def parse_captions(value):

    if value is None:
        return []

    if isinstance(
        value,
        float
    ) and np.isnan(value):

        return []

    if isinstance(
        value,
        list
    ):

        return [
            str(x).strip()
            for x in value
            if str(x).strip()
        ]

    value = str(value).strip()

    if not value:
        return []

    try:

        parsed = json.loads(
            value
        )

        if isinstance(
            parsed,
            list
        ):

            return [
                str(x).strip()
                for x in parsed
                if str(x).strip()
            ]

    except Exception:
        pass

    try:

        parsed = ast.literal_eval(
            value
        )

        if isinstance(
            parsed,
            list
        ):

            return [
                str(x).strip()
                for x in parsed
                if str(x).strip()
            ]

    except Exception:
        pass

    return [value]


# ================================================================
# 10. TEXT TOKENIZATION
# ================================================================

def tokenize(text):

    text = str(text).lower()

    # Keep words and basic punctuation
    tokens = re.findall(
        r"[a-z0-9]+",
        text
    )

    return tokens


# ================================================================
# 11. BUILD VOCABULARY
# ================================================================

PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
UNK_TOKEN = "<UNK>"

special_tokens = [
    PAD_TOKEN,
    SOS_TOKEN,
    EOS_TOKEN,
    UNK_TOKEN
]

word_frequency = Counter()

for captions in train_df["captions"]:

    captions = parse_captions(
        captions
    )

    for caption in captions:

        tokens = tokenize(
            caption
        )

        word_frequency.update(
            tokens
        )


vocabulary_words = [
    word
    for word, count
    in word_frequency.items()
    if count >= MIN_WORD_FREQ
]

vocabulary_words = sorted(
    vocabulary_words
)

itos = (
    special_tokens
    + vocabulary_words
)

stoi = {
    word: idx
    for idx, word in enumerate(
        itos
    )
}

PAD_IDX = stoi[
    PAD_TOKEN
]

SOS_IDX = stoi[
    SOS_TOKEN
]

EOS_IDX = stoi[
    EOS_TOKEN
]

UNK_IDX = stoi[
    UNK_TOKEN
]

VOCAB_SIZE = len(
    itos
)

print()
print(
    f"Vocabulary size: {VOCAB_SIZE}"
)


# ================================================================
# 12. SAVE VOCABULARY
# ================================================================

with open(
    VOCAB_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "stoi": stoi,
            "itos": itos,
            "pad_idx": PAD_IDX,
            "sos_idx": SOS_IDX,
            "eos_idx": EOS_IDX,
            "unk_idx": UNK_IDX
        },
        f,
        indent=2
    )


# ================================================================
# 13. ENCODE CAPTION
# ================================================================

def encode_caption(
    caption
):

    tokens = tokenize(
        caption
    )

    token_ids = [
        SOS_IDX
    ]

    for token in tokens:

        token_ids.append(
            stoi.get(
                token,
                UNK_IDX
            )
        )

    token_ids.append(
        EOS_IDX
    )

    token_ids = token_ids[
        :MAX_CAPTION_LEN
    ]

    if len(token_ids) < MAX_CAPTION_LEN:

        token_ids += (
            [PAD_IDX]
            *
            (
                MAX_CAPTION_LEN
                -
                len(token_ids)
            )
        )

    return torch.tensor(
        token_ids,
        dtype=torch.long
    )


# ================================================================
# 14. PATH RESOLUTION
# ================================================================

BASE_DIR = Path(
    r"C:\College\Dissertation\Data\FINAL_LEVIR_DATASET"
)


def resolve_path(value):

    if pd.isna(value):

        return None

    path = Path(
        str(value)
    )

    if path.is_absolute():

        if path.exists():
            return path

    candidate = (
        BASE_DIR / path
    )

    if candidate.exists():
        return candidate

    if path.exists():
        return path

    return None


# ================================================================
# 15. DATASET
# ================================================================

class CaptionDataset(
    Dataset
):

    def __init__(
        self,
        dataframe
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transforms.Compose([

            transforms.Resize(
                (
                    IMAGE_SIZE,
                    IMAGE_SIZE
                )
            ),

            transforms.ToTensor(),

            transforms.Normalize(
                mean=[
                    0.485,
                    0.456,
                    0.406
                ],

                std=[
                    0.229,
                    0.224,
                    0.225
                ]
            )
        ])


    def __len__(self):

        return len(
            self.df
        )


    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[
            index
        ]

        path_A = resolve_path(
            row["cc_A"]
        )

        path_B = resolve_path(
            row["cc_B"]
        )

        if path_A is None:

            raise FileNotFoundError(
                f"Image A not found:\n"
                f"{row['cc_A']}"
            )

        if path_B is None:

            raise FileNotFoundError(
                f"Image B not found:\n"
                f"{row['cc_B']}"
            )

        image_A = Image.open(
            path_A
        ).convert("RGB")

        image_B = Image.open(
            path_B
        ).convert("RGB")

        image_A = self.transform(
            image_A
        )

        image_B = self.transform(
            image_B
        )

        captions = parse_captions(
            row["captions"]
        )

        # Use first caption as the training target.
        target_caption = captions[0]

        encoded_caption = encode_caption(
            target_caption
        )

        return {
            "image_A": image_A,
            "image_B": image_B,
            "caption": encoded_caption,
            "captions": captions,
            "changeflag": int(
                row["changeflag"]
            )
        }


# ================================================================
# 16. IMPORT TRANSFORMS
# ================================================================

from torchvision import transforms


# ================================================================
# 17. DATA LOADERS
# ================================================================

train_dataset = CaptionDataset(
    train_df
)

val_dataset = CaptionDataset(
    val_df
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(
        DEVICE.type == "cuda"
    )
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(
        DEVICE.type == "cuda"
    )
)

print()
print(
    f"Train batches: "
    f"{len(train_loader)}"
)

print(
    f"Validation batches: "
    f"{len(val_loader)}"
)


# ================================================================
# 18. SIAMESE RESNET18 ENCODER
# ================================================================

class SiameseResNet18Encoder(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        backbone = models.resnet18(
            weights=ResNet18_Weights.DEFAULT
        )

        self.encoder = nn.Sequential(
            *list(
                backbone.children()
            )[:-2]
        )

        # ResNet18 final convolutional
        # feature map:
        #
        # [B, 512, 7, 7]
        #
        # for 224x224 input.


    def forward(
        self,
        image_A,
        image_B
    ):

        feature_A = self.encoder(
            image_A
        )

        feature_B = self.encoder(
            image_B
        )

        difference = torch.abs(
            feature_A
            -
            feature_B
        )

        # Temporal information:
        #
        # A
        # B
        # |A-B|
        #
        combined = torch.cat(
            [
                feature_A,
                feature_B,
                difference
            ],
            dim=1
        )

        return combined


# ================================================================
# 19. CAPTION DECODER
# ================================================================

class CaptionDecoder(
    nn.Module
):

    def __init__(
        self,
        feature_dim,
        vocab_size,
        hidden_dim=256
    ):

        super().__init__()

        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(
            vocab_size,
            hidden_dim,
            padding_idx=PAD_IDX
        )

        self.feature_projection = nn.Linear(
            feature_dim,
            hidden_dim
        )

        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.output = nn.Linear(
            hidden_dim,
            vocab_size
        )


    def forward(
        self,
        features,
        captions
    ):

        # --------------------------------------------------------
        # Global temporal feature
        # --------------------------------------------------------

        features = F.adaptive_avg_pool2d(
            features,
            1
        )

        features = features.flatten(
            1
        )

        hidden = self.feature_projection(
            features
        )

        hidden = hidden.unsqueeze(
            0
        )

        # Teacher forcing
        inputs = captions[:, :-1]

        embeddings = self.embedding(
            inputs
        )

        outputs, _ = self.gru(
            embeddings,
            hidden
        )

        logits = self.output(
            outputs
        )

        return logits


# ================================================================
# 20. COMPLETE MODEL
# ================================================================

class ChangeAwareCaptionModel(
    nn.Module
):

    def __init__(
        self,
        vocab_size
    ):

        super().__init__()

        self.encoder = (
            SiameseResNet18Encoder()
        )

        self.decoder = CaptionDecoder(
            feature_dim=1536,
            vocab_size=vocab_size,
            hidden_dim=HIDDEN_DIM
        )


    def forward(
        self,
        image_A,
        image_B,
        captions
    ):

        features = self.encoder(
            image_A,
            image_B
        )

        logits = self.decoder(
            features,
            captions
        )

        return logits


# ================================================================
# 21. INITIALIZE MODEL
# ================================================================

model = ChangeAwareCaptionModel(
    VOCAB_SIZE
).to(
    DEVICE
)

parameter_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print()
print(
    f"Trainable parameters: "
    f"{parameter_count / 1e6:.2f}M"
)


# ================================================================
# 22. SHAPE CHECK
# ================================================================

print()
print("=" * 72)
print("MODEL SHAPE CHECK")
print("=" * 72)

sample_batch = next(
    iter(train_loader)
)

sample_A = sample_batch[
    "image_A"
][:2].to(DEVICE)

sample_B = sample_batch[
    "image_B"
][:2].to(DEVICE)

sample_caption = sample_batch[
    "caption"
][:2].to(DEVICE)

model.eval()

with torch.no_grad():

    sample_output = model(
        sample_A,
        sample_B,
        sample_caption
    )

print(
    "Input A:",
    sample_A.shape
)

print(
    "Input B:",
    sample_B.shape
)

print(
    "Caption:",
    sample_caption.shape
)

print(
    "Output:",
    sample_output.shape
)

expected_shape = (
    2,
    MAX_CAPTION_LEN - 1,
    VOCAB_SIZE
)

assert sample_output.shape == expected_shape, (
    f"Unexpected output shape: "
    f"{sample_output.shape}"
)

print(
    "MODEL SHAPE CHECK: PASS"
)

del sample_batch
del sample_A
del sample_B
del sample_caption
del sample_output

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


# ================================================================
# 23. LOSS
# ================================================================

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)


# ================================================================
# 24. OPTIMIZER
# ================================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3,
    min_lr=1e-7
)


# ================================================================
# 25. AMP
# ================================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(
        DEVICE.type == "cuda"
    )
)


# ================================================================
# 26. TRAINING FUNCTION
# ================================================================

def train_one_epoch():

    model.train()

    running_loss = 0.0

    progress = tqdm(
        train_loader,
        desc="Training"
    )

    for batch in progress:

        image_A = batch[
            "image_A"
        ].to(
            DEVICE,
            non_blocking=True
        )

        image_B = batch[
            "image_B"
        ].to(
            DEVICE,
            non_blocking=True
        )

        captions = batch[
            "caption"
        ].to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            enabled=(
                DEVICE.type == "cuda"
            )
        ):

            logits = model(
                image_A,
                image_B,
                captions
            )

            targets = captions[:, 1:]

            loss = criterion(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                targets.reshape(
                    -1
                )
            )

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        running_loss += (
            loss.item()
        )

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return (
        running_loss
        /
        len(train_loader)
    )


# ================================================================
# 27. VALIDATION LOSS
# ================================================================

@torch.no_grad()
def validate_loss():

    model.eval()

    running_loss = 0.0

    progress = tqdm(
        val_loader,
        desc="Validation"
    )

    for batch in progress:

        image_A = batch[
            "image_A"
        ].to(
            DEVICE,
            non_blocking=True
        )

        image_B = batch[
            "image_B"
        ].to(
            DEVICE,
            non_blocking=True
        )

        captions = batch[
            "caption"
        ].to(
            DEVICE,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            enabled=(
                DEVICE.type == "cuda"
            )
        ):

            logits = model(
                image_A,
                image_B,
                captions
            )

            targets = captions[:, 1:]

            loss = criterion(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                targets.reshape(
                    -1
                )
            )

        running_loss += (
            loss.item()
        )

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return (
        running_loss
        /
        len(val_loader)
    )


# ================================================================
# 28. GREEDY CAPTION GENERATION
# ================================================================

@torch.no_grad()
def generate_caption(
    image_A,
    image_B
):

    model.eval()

    features = model.encoder(
        image_A,
        image_B
    )

    features = F.adaptive_avg_pool2d(
        features,
        1
    )

    features = features.flatten(
        1
    )

    hidden = model.decoder.feature_projection(
        features
    ).unsqueeze(0)

    current_token = torch.tensor(
        [SOS_IDX],
        dtype=torch.long,
        device=DEVICE
    )

    generated_tokens = []

    for _ in range(
        MAX_CAPTION_LEN - 1
    ):

        embedding = model.decoder.embedding(
            current_token
        ).unsqueeze(1)

        output, hidden = model.decoder.gru(
            embedding,
            hidden
        )

        logits = model.decoder.output(
            output.squeeze(1)
        )

        next_token = torch.argmax(
            logits,
            dim=1
        )

        token_id = int(
            next_token.item()
        )

        if token_id == EOS_IDX:

            break

        if token_id not in [
            PAD_IDX,
            SOS_IDX
        ]:

            generated_tokens.append(
                token_id
            )

        current_token = next_token

    words = []

    for token_id in generated_tokens:

        if (
            0 <= token_id
            < len(itos)
        ):

            word = itos[
                token_id
            ]

            if word not in [
                PAD_TOKEN,
                SOS_TOKEN,
                EOS_TOKEN
            ]:

                words.append(
                    word
                )

    return " ".join(
        words
    )


# ================================================================
# 29. TRAINING LOOP
# ================================================================

print()
print("=" * 72)
print("STARTING MODEL 3 TRAINING")
print("=" * 72)

best_val_loss = float(
    "inf"
)

best_epoch = 0

epochs_without_improvement = 0

history = []

training_start = time.time()


for epoch in range(
    1,
    EPOCHS + 1
):

    epoch_start = time.time()

    train_loss = (
        train_one_epoch()
    )

    val_loss = (
        validate_loss()
    )

    scheduler.step(
        val_loss
    )

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )

    elapsed = (
        time.time()
        -
        epoch_start
    )

    history.append({

        "epoch": epoch,

        "train_loss": train_loss,

        "val_loss": val_loss,

        "learning_rate": current_lr,

        "epoch_time_sec": elapsed
    })

    print()
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train: {train_loss:.4f} | "
        f"Val: {val_loss:.4f} | "
        f"LR: {current_lr:.2e} | "
        f"{elapsed:.1f}s"
    )


    # ------------------------------------------------------------
    # BEST MODEL
    # ------------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_epoch = epoch

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "scheduler_state_dict":
                    scheduler.state_dict(),

                "best_val_loss":
                    best_val_loss,

                "vocab_size":
                    VOCAB_SIZE,

                "stoi":
                    stoi,

                "itos":
                    itos,

                "config": {
                    "image_size":
                        IMAGE_SIZE,

                    "hidden_dim":
                        HIDDEN_DIM,

                    "max_caption_len":
                        MAX_CAPTION_LEN,

                    "batch_size":
                        BATCH_SIZE,

                    "learning_rate":
                        LEARNING_RATE
                }
            },
            BEST_MODEL_PATH
        )

        print(
            f"    ★ BEST MODEL SAVED "
            f"| Val Loss = "
            f"{best_val_loss:.4f}"
        )

    else:

        epochs_without_improvement += 1


    # ------------------------------------------------------------
    # SAVE HISTORY
    # ------------------------------------------------------------

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False
    )


    # ------------------------------------------------------------
    # EARLY STOPPING
    # ------------------------------------------------------------

    if (
        epochs_without_improvement
        >= PATIENCE
    ):

        print()
        print(
            "Early stopping triggered."
        )

        break


# ================================================================
# 30. TRAINING COMPLETE
# ================================================================

total_training_time = (
    time.time()
    -
    training_start
)

print()
print("=" * 72)
print("TRAINING COMPLETE")
print("=" * 72)

print(
    f"Best epoch: {best_epoch}"
)

print(
    f"Best validation loss: "
    f"{best_val_loss:.6f}"
)

print(
    f"Training time: "
    f"{total_training_time / 60:.2f} minutes"
)


# ================================================================
# 31. LOAD BEST MODEL
# ================================================================

print()
print("=" * 72)
print("LOADING BEST MODEL")
print("=" * 72)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

print(
    f"Loaded epoch: "
    f"{checkpoint['epoch']}"
)

print(
    f"Validation loss: "
    f"{checkpoint['best_val_loss']:.6f}"
)


# ================================================================
# 32. EVALUATION
# ================================================================

print()
print("=" * 72)
print("GENERATING VALIDATION CAPTIONS")
print("=" * 72)


predictions = []

all_references = []

change_predictions = []

no_change_predictions = []


for index in tqdm(
    range(
        len(val_dataset)
    ),
    desc="Caption generation"
):

    sample = val_dataset[
        index
    ]

    image_A = sample[
        "image_A"
    ].unsqueeze(0).to(
        DEVICE
    )

    image_B = sample[
        "image_B"
    ].unsqueeze(0).to(
        DEVICE
    )

    generated = generate_caption(
        image_A,
        image_B
    )

    references = [
        tokenize(
            caption
        )
        for caption
        in sample["captions"]
    ]

    prediction_tokens = tokenize(
        generated
    )

    predictions.append(
        generated
    )

    all_references.append(
        references
    )

    record = {
        "index": index,

        "changeflag":
            sample["changeflag"],

        "prediction":
            generated,

        "reference_1":
            sample["captions"][0]
            if len(
                sample["captions"]
            ) > 0 else "",

        "reference_2":
            sample["captions"][1]
            if len(
                sample["captions"]
            ) > 1 else "",

        "reference_3":
            sample["captions"][2]
            if len(
                sample["captions"]
            ) > 2 else "",

        "reference_4":
            sample["captions"][3]
            if len(
                sample["captions"]
            ) > 3 else "",

        "reference_5":
            sample["captions"][4]
            if len(
                sample["captions"]
            ) > 4 else ""
    }

    if sample[
        "changeflag"
    ] == 1:

        change_predictions.append(
            (
                references,
                prediction_tokens
            )
        )

    else:

        no_change_predictions.append(
            (
                references,
                prediction_tokens
            )
        )


# ================================================================
# 33. BLEU
# ================================================================

smooth = (
    SmoothingFunction()
    .method1
)


bleu1 = corpus_bleu(
    all_references,
    predictions=[
        tokenize(x)
        for x in predictions
    ],
    weights=(
        1.0,
        0,
        0,
        0
    ),
    smoothing_function=smooth
)


bleu2 = corpus_bleu(
    all_references,
    predictions=[
        tokenize(x)
        for x in predictions
    ],
    weights=(
        0.5,
        0.5,
        0,
        0
    ),
    smoothing_function=smooth
)


bleu3 = corpus_bleu(
    all_references,
    predictions=[
        tokenize(x)
        for x in predictions
    ],
    weights=(
        1 / 3,
        1 / 3,
        1 / 3,
        0
    ),
    smoothing_function=smooth
)


bleu4 = corpus_bleu(
    all_references,
    predictions=[
        tokenize(x)
        for x in predictions
    ],
    weights=(
        0.25,
        0.25,
        0.25,
        0.25
    ),
    smoothing_function=smooth
)


# ================================================================
# 34. METEOR
# ================================================================

meteor_scores = []

for references, prediction in zip(
    all_references,
    [
        tokenize(x)
        for x in predictions
    ]
):

    meteor_scores.append(
        meteor_score(
            references,
            prediction
        )
    )


meteor = np.mean(
    meteor_scores
)


# ================================================================
# 35. ROUGE-L
# ================================================================

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

rouge_scores = []


for sample_index in range(
    len(predictions)
):

    prediction = predictions[
        sample_index
    ]

    references = val_dataset[
        sample_index
    ]["captions"]

    best_score = 0.0

    for reference in references:

        score = rouge.score(
            reference,
            prediction
        )["rougeL"].fmeasure

        best_score = max(
            best_score,
            score
        )

    rouge_scores.append(
        best_score
    )


rouge_l = np.mean(
    rouge_scores
)


# ================================================================
# 36. PRINT FINAL RESULTS
# ================================================================

print()
print("=" * 72)
print("MODEL 3 — FINAL VALIDATION RESULTS")
print("=" * 72)

print(
    f"BLEU-1  : {bleu1:.4f}"
)

print(
    f"BLEU-2  : {bleu2:.4f}"
)

print(
    f"BLEU-3  : {bleu3:.4f}"
)

print(
    f"BLEU-4  : {bleu4:.4f}"
)

print(
    f"METEOR  : {meteor:.4f}"
)

print(
    f"ROUGE-L : {rouge_l:.4f}"
)


# ================================================================
# 37. CHANGE / NO-CHANGE BREAKDOWN
# ================================================================

def calculate_subset_metrics(
    subset
):

    if len(subset) == 0:

        return {
            "BLEU1": 0,
            "BLEU2": 0,
            "BLEU3": 0,
            "BLEU4": 0,
            "METEOR": 0,
            "ROUGE_L": 0
        }

    refs = [
        x[0]
        for x in subset
    ]

    preds = [
        x[1]
        for x in subset
    ]

    b1 = corpus_bleu(
        refs,
        preds,
        weights=(
            1, 0, 0, 0
        ),
        smoothing_function=smooth
    )

    b2 = corpus_bleu(
        refs,
        preds,
        weights=(
            0.5,
            0.5,
            0,
            0
        ),
        smoothing_function=smooth
    )

    b3 = corpus_bleu(
        refs,
        preds,
        weights=(
            1/3,
            1/3,
            1/3,
            0
        ),
        smoothing_function=smooth
    )

    b4 = corpus_bleu(
        refs,
        preds,
        weights=(
            0.25,
            0.25,
            0.25,
            0.25
        ),
        smoothing_function=smooth
    )

    meteor_values = []

    for reference, prediction in zip(
        refs,
        preds
    ):

        meteor_values.append(
            meteor_score(
                reference,
                prediction
            )
        )

    meteor_value = np.mean(
        meteor_values
    )

    return {
        "BLEU1": b1,
        "BLEU2": b2,
        "BLEU3": b3,
        "BLEU4": b4,
        "METEOR": meteor_value
    }


change_metrics = (
    calculate_subset_metrics(
        change_predictions
    )
)

no_change_metrics = (
    calculate_subset_metrics(
        no_change_predictions
    )
)


# ================================================================
# 38. PRINT BREAKDOWN
# ================================================================

print()
print("=" * 72)
print("CHANGE / NO-CHANGE CAPTION PERFORMANCE")
print("=" * 72)

print()
print(
    "CHANGE SAMPLES:",
    len(change_predictions)
)

print(
    f"BLEU-1 : "
    f"{change_metrics['BLEU1']:.4f}"
)

print(
    f"BLEU-2 : "
    f"{change_metrics['BLEU2']:.4f}"
)

print(
    f"BLEU-3 : "
    f"{change_metrics['BLEU3']:.4f}"
)

print(
    f"BLEU-4 : "
    f"{change_metrics['BLEU4']:.4f}"
)

print(
    f"METEOR : "
    f"{change_metrics['METEOR']:.4f}"
)

print()
print(
    "NO-CHANGE SAMPLES:",
    len(no_change_predictions)
)

print(
    f"BLEU-1 : "
    f"{no_change_metrics['BLEU1']:.4f}"
)

print(
    f"BLEU-2 : "
    f"{no_change_metrics['BLEU2']:.4f}"
)

print(
    f"BLEU-3 : "
    f"{no_change_metrics['BLEU3']:.4f}"
)

print(
    f"BLEU-4 : "
    f"{no_change_metrics['BLEU4']:.4f}"
)

print(
    f"METEOR : "
    f"{no_change_metrics['METEOR']:.4f}"
)


# ================================================================
# 39. SAVE PREDICTIONS
# ================================================================

results_df = pd.DataFrame(
    [
        {
            "index": i,

            "changeflag":
                val_df.iloc[i][
                    "changeflag"
                ],

            "prediction":
                predictions[i],

            "reference_1":
                val_dataset[i][
                    "captions"
                ][0]
                if len(
                    val_dataset[i]["captions"]
                ) > 0 else "",

            "reference_2":
                val_dataset[i][
                    "captions"
                ][1]
                if len(
                    val_dataset[i]["captions"]
                ) > 1 else "",

            "reference_3":
                val_dataset[i][
                    "captions"
                ][2]
                if len(
                    val_dataset[i]["captions"]
                ) > 2 else "",

            "reference_4":
                val_dataset[i][
                    "captions"
                ][3]
                if len(
                    val_dataset[i]["captions"]
                ) > 3 else "",

            "reference_5":
                val_dataset[i][
                    "captions"
                ][4]
                if len(
                    val_dataset[i]["captions"]
                ) > 4 else ""
        }

        for i in range(
            len(predictions)
        )
    ]
)

results_df.to_csv(
    RESULTS_PATH,
    index=False
)


# ================================================================
# 40. SAVE REPORT
# ================================================================

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "MODEL 3 — CHANGE-AWARE IMAGE CAPTIONING\n"
    )

    f.write(
        "=" * 72 + "\n\n"
    )

    f.write(
        "Overall Validation Metrics\n"
    )

    f.write(
        "-" * 72 + "\n"
    )

    f.write(
        f"BLEU-1  : {bleu1:.6f}\n"
    )

    f.write(
        f"BLEU-2  : {bleu2:.6f}\n"
    )

    f.write(
        f"BLEU-3  : {bleu3:.6f}\n"
    )

    f.write(
        f"BLEU-4  : {bleu4:.6f}\n"
    )

    f.write(
        f"METEOR  : {meteor:.6f}\n"
    )

    f.write(
        f"ROUGE-L : {rouge_l:.6f}\n"
    )

    f.write(
        "\n\nChange Samples\n"
    )

    f.write(
        "-" * 72 + "\n"
    )

    for key, value in change_metrics.items():

        f.write(
            f"{key}: {value:.6f}\n"
        )

    f.write(
        "\n\nNo-Change Samples\n"
    )

    f.write(
        "-" * 72 + "\n"
    )

    for key, value in no_change_metrics.items():

        f.write(
            f"{key}: {value:.6f}\n"
        )


# ================================================================
# 41. SAVE CONFIG
# ================================================================

config = {

    "task":
        "Change-Aware Image Captioning",

    "dataset":
        "LEVIR-CC",

    "architecture":
        "Siamese ResNet18 + GRU",

    "temporal_representation":
        "A + B + absolute difference",

    "image_size":
        IMAGE_SIZE,

    "batch_size":
        BATCH_SIZE,

    "epochs":
        EPOCHS,

    "best_epoch":
        best_epoch,

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "hidden_dim":
        HIDDEN_DIM,

    "vocabulary_size":
        VOCAB_SIZE,

    "max_caption_length":
        MAX_CAPTION_LEN,

    "best_validation_loss":
        float(best_val_loss),

    "BLEU_1":
        float(bleu1),

    "BLEU_2":
        float(bleu2),

    "BLEU_3":
        float(bleu3),

    "BLEU_4":
        float(bleu4),

    "METEOR":
        float(meteor),

    "ROUGE_L":
        float(rouge_l)
}

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )


# ================================================================
# 42. QUALITATIVE EXAMPLES
# ================================================================

print()
print("=" * 72)
print("QUALITATIVE EXAMPLES")
print("=" * 72)

for i in range(
    min(15, len(predictions))
):

    print()
    print(
        f"Example {i + 1}"
    )

    print(
        "Prediction:",
        predictions[i]
    )

    print(
        "Ground Truth:",
        val_dataset[i][
            "captions"
        ][0]
    )


# ================================================================
# 43. FINAL OUTPUT
# ================================================================

print()
print("=" * 72)
print("MODEL 3 COMPLETE")
print("=" * 72)

print()
print(
    f"Best checkpoint:\n"
    f"{BEST_MODEL_PATH}"
)

print()
print(
    f"Vocabulary:\n"
    f"{VOCAB_PATH}"
)

print()
print(
    f"Predictions:\n"
    f"{RESULTS_PATH}"
)

print()
print(
    f"Evaluation report:\n"
    f"{REPORT_PATH}"
)

print()
print(
    f"Training history:\n"
    f"{HISTORY_PATH}"
)

print()
print(
    f"Configuration:\n"
    f"{CONFIG_PATH}"
)

print()
print("=" * 72)

[nltk_data] Downloading package wordnet to C:\Users\Aishwarya
[nltk_data]     Gilhotra\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Aishwarya
[nltk_data]     Gilhotra\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


MODEL 3 — CHANGE-AWARE IMAGE CAPTIONING
Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
GPU Memory: 7.96 GB

LOADING DATASET
Total rows: 7,264
split
train    6345
val       919
Name: count, dtype: int64
Unique CC pairs: 7,264
Duplicates removed: 0
Train: 6,345
Validation: 919

Vocabulary size: 852

Train batches: 397
Validation batches: 58

Trainable parameters: 12.40M

MODEL SHAPE CHECK
Input A: torch.Size([2, 3, 224, 224])
Input B: torch.Size([2, 3, 224, 224])
Caption: torch.Size([2, 40])
Output: torch.Size([2, 39, 852])
MODEL SHAPE CHECK: PASS

STARTING MODEL 3 TRAINING


Training:   0%|          | 0/397 [00:00<?, ?it/s]

Validation:   0%|          | 0/58 [00:00<?, ?it/s]


Epoch 01/20 | Train: 2.8835 | Val: 1.8343 | LR: 1.00e-04 | 80.3s
    ★ BEST MODEL SAVED | Val Loss = 1.8343


...

Training:   0%|          | 0/397 [00:00<?, ?it/s]

Validation:   0%|          | 0/58 [00:00<?, ?it/s]


Epoch 12/20 | Train: 0.4528 | Val: 1.2933 | LR: 5.00e-05 | 68.5s

Early stopping triggered.

TRAINING COMPLETE
Best epoch: 6
Best validation loss: 1.226821
Training time: 13.93 minutes

LOADING BEST MODEL
Loaded epoch: 6
Validation loss: 1.226821

GENERATING VALIDATION CAPTIONS


Caption generation:   0%|          | 0/919 [00:00<?, ?it/s]

TypeError: corpus_bleu() got an unexpected keyword argument 'predictions'

In [2]:
# ================================================================
# FIXED BLEU EVALUATION
# ================================================================

candidate_tokens = [
    tokenize(x)
    for x in predictions
]

bleu1 = corpus_bleu(
    all_references,
    candidate_tokens,
    weights=(1.0, 0, 0, 0),
    smoothing_function=smooth
)

bleu2 = corpus_bleu(
    all_references,
    candidate_tokens,
    weights=(0.5, 0.5, 0, 0),
    smoothing_function=smooth
)

bleu3 = corpus_bleu(
    all_references,
    candidate_tokens,
    weights=(1/3, 1/3, 1/3, 0),
    smoothing_function=smooth
)

bleu4 = corpus_bleu(
    all_references,
    candidate_tokens,
    weights=(0.25, 0.25, 0.25, 0.25),
    smoothing_function=smooth
)

# ================================================================
# METEOR
# ================================================================

meteor_scores = []

for references, prediction in zip(
    all_references,
    candidate_tokens
):

    meteor_scores.append(
        meteor_score(
            references,
            prediction
        )
    )

meteor = np.mean(
    meteor_scores
)

# ================================================================
# ROUGE-L
# ================================================================

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

rouge_scores = []

for i in range(
    len(predictions)
):

    prediction = predictions[i]

    references = val_dataset[i]["captions"]

    best_score = 0.0

    for reference in references:

        score = rouge.score(
            reference,
            prediction
        )["rougeL"].fmeasure

        best_score = max(
            best_score,
            score
        )

    rouge_scores.append(
        best_score
    )

rouge_l = np.mean(
    rouge_scores
)

# ================================================================
# FINAL RESULTS
# ================================================================

print()
print("=" * 72)
print("MODEL 3 — FINAL VALIDATION RESULTS")
print("=" * 72)

print(f"BLEU-1  : {bleu1:.4f}")
print(f"BLEU-2  : {bleu2:.4f}")
print(f"BLEU-3  : {bleu3:.4f}")
print(f"BLEU-4  : {bleu4:.4f}")
print(f"METEOR  : {meteor:.4f}")
print(f"ROUGE-L : {rouge_l:.4f}")


MODEL 3 — FINAL VALIDATION RESULTS
BLEU-1  : 0.6959
BLEU-2  : 0.5823
BLEU-3  : 0.4853
BLEU-4  : 0.3920
METEOR  : 0.6472
ROUGE-L : 0.6607
